In [1]:
import os
import numpy as np
import community as community
import random
from tqdm import tqdm, trange
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn import metrics


In [2]:
def ale_1d(model, X, feature, edges, class_index=0):
    """
    Compute 1D Accumulated Local Effects (ALE) for a single feature on
    predicted class probability (classification). Uses fixed bin edges.

    Parameters
    ----------
    model : fitted classifier with predict_proba
    X : pandas.DataFrame (data used to estimate the ALE; e.g., test or train)
    feature : str
    edges : np.array of shape (n_bins+1,)
    class_index : int (which class's probability to explain)

    Returns
    -------
    pandas.DataFrame with columns:
      - 'x': bin midpoints
      - 'ale': centered accumulated local effect per bin
      - 'count': number of samples in each bin
    """
    X = X.copy()
    n_bins = len(edges) - 1
    local_effects = []
    counts = []

    for k in range(n_bins):
        lower, upper = edges[k], edges[k+1]
        # last bin inclusive on upper bound to capture max
        if k == n_bins - 1:
            mask = (X[feature] >= lower) & (X[feature] <= upper)
        else:
            mask = (X[feature] >= lower) & (X[feature] < upper)

        n_k = int(mask.sum())
        counts.append(n_k)

        if n_k == 0:
            local_effects.append(0.0)
            continue

        # Create perturbed datasets
        X_upper = X.loc[mask].copy()
        X_lower = X.loc[mask].copy()
        X_upper[feature] = upper
        X_lower[feature] = lower

        # Class probability at bin edges
        p_upper = model.predict_proba(X_upper)[:, class_index]
        p_lower = model.predict_proba(X_lower)[:, class_index]

        # Average local effect in this bin
        local_effects.append((p_upper - p_lower).mean())

    # Accumulate local effects to get ALE curve
    ale = np.cumsum(local_effects)
    midpoints = (edges[:-1] + edges[1:]) / 2.0

    # Center the ALE curve (weighted by bin counts, standard in ALE)
    weights = np.array(counts, dtype=float)
    wsum = weights.sum()
    mean_ale = (ale * weights).sum() / (wsum if wsum > 0 else 1.0)
    ale_centered = ale - mean_ale

    return pd.DataFrame({"x": midpoints, "ale": ale_centered, "count": counts})


In [3]:
RUNS = 10

def node_dataset_gen(X, entropy_values, us):
    kmeans_seed = random.randint(0, 10000)
    kmeans = KMeans(n_clusters=2, random_state=kmeans_seed).fit(entropy_values.reshape(-1, 1))
    cutoff = np.mean(kmeans.cluster_centers_)
    y = np.where(entropy_values < cutoff, 0, 1)
    if us != None:
        stab_unstab = np.bincount(y)
        num_unstab = stab_unstab[1]
        num_stab = int(num_unstab / us)
        lowest_entropy_indices = list(entropy_values.argsort()[:num_stab][::-1])
        highest_entropy_indices = list(entropy_values.argsort()[-num_unstab:][::-1])
        X_lowest = X.iloc[lowest_entropy_indices]
        X_highest = X.iloc[highest_entropy_indices]
        X = pd.concat([X_lowest, X_highest])
        y = [0 for _ in range(num_stab)] + [1 for _ in range(num_unstab)]
        y = pd.DataFrame(y, index=X.index, columns=['Stability'])
    else:
        y = pd.DataFrame(y, index=X.index, columns=['Stability'])
    split_seed = random.randint(0, 10000)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=split_seed)
    return X_train, X_test, y_train, y_test, cutoff


def pair_dataset_gen(X, entropy_values):
    lowest_entropy_indices = list(entropy_values.argsort()[:500][::-1])
    highest_entropy_indices = list(entropy_values.argsort()[-500:][::-1])
    X_lowest = X.iloc[lowest_entropy_indices]
    X_highest = X.iloc[highest_entropy_indices]
    X = pd.concat([X_lowest, X_highest])
    y = [0 for _ in range(500)] + [1 for _ in range(500)]
    y = pd.DataFrame(y, index=X.index, columns=['Same Community'])
    split_seed = random.randint(0, 10000)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=split_seed)
    return X_train, X_test, y_train, y_test


def save_dataset(X_train, X_test, y_train, y_test, graph_name):
    """
    Salva os datasets com nome único por grafo para evitar sobrescrita.
    """
    X_train.to_csv(os.path.join(results_folder, f'{graph_name}_X_train.csv'))
    y_train.to_csv(os.path.join(results_folder, f'{graph_name}_y_train.csv'))
    X_test.to_csv(os.path.join(results_folder, f'{graph_name}_X_test.csv'))
    y_test.to_csv(os.path.join(results_folder, f'{graph_name}_y_test.csv'))
    return

def create_and_save_data(feats_fil, entropies_fil, results_folder, us, mode, graph_name):
    """
    Cria os datasets e salva com nome único por grafo.
    """
    X = pd.read_csv(feats_fil, index_col=0)
    entrops = pd.read_csv(entropies_fil, index_col=0)
    entropy_values = np.array(entrops['Entropy'])
    entropy_values = entropy_values.reshape(-1)
    
    if mode == 'node':
        X_train, X_test, y_train, y_test, cutoff = node_dataset_gen(X, entropy_values, us)
    elif mode == 'pair':
        X_train, X_test, y_train, y_test = pair_dataset_gen(X, entropy_values)
    
    # Salva com nome único
    save_dataset(X_train, X_test, y_train, y_test, graph_name)
    
    if mode == 'node':
        label_counts = np.bincount(y_train['Stability'])
        results_dict = {
            'Stable Nodes': label_counts[0], 
            'Unstable Nodes': label_counts[1],
            'Stability Cutoff': cutoff, 
            'Undersampling Level': us
        }
    elif mode == 'pair':
        label_counts = np.bincount(y_train['Same Community'])
        results_dict = {
            'Different Communities': label_counts[0], 
            'Same Communities': label_counts[1],
            'Undersampling Level': us
        }
    
    return X_train, y_train, X_test, y_test, results_dict



def train(X_train, X_test, y_train, y_test, n_splits=5, n_bins=20):
    """
    Train RandomForest with Stratified CV and compute 1D ALE curves per feature.

    Returns
    -------
    ale_curves : dict[str, pandas.DataFrame]
        For each feature, a DataFrame with columns:
            - 'x' (bin midpoints),
            - 'ale_mean',
            - 'ale_std',
            - 'avg_bin_count'
    accuracy_scores : list[float]
    balanced_accuracy_scores : list[float]
    """
    feature_list = list(X_train.columns)
    data = np.array(X_train)
    labels = np.squeeze(np.array(y_train))
    accuracy_scores = []
    balanced_accuracy_scores = []

    # Precompute fixed bin edges from X_test quantiles (stable across folds)
    quantiles = np.linspace(0, 1, n_bins + 1)
    feature_edges = {}
    for f in feature_list:
        edges = np.quantile(X_test[f], quantiles)
        # Ensure strictly increasing edges; fallback to linspace if too many duplicates
        edges_unique = np.unique(edges)
        if len(edges_unique) < 2:
            mn = float(X_test[f].min())
            mx = float(X_test[f].max())
            edges_unique = np.linspace(mn, mx, n_bins + 1)
        feature_edges[f] = edges_unique

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    rf = RandomForestClassifier(random_state=42)

    # Collect+ ALE curves per feature across folds/runs
    ale_stack = {f: [] for f in feature_list}

    fold_count = 0
    for run in trange(1, RUNS + 1):
        for train_idx, val_idx in skf.split(data, labels):
            fold_count += 1

            X_train_fold = X_train.iloc[train_idx]
            X_val_fold   = X_train.iloc[val_idx]
            if hasattr(y_train, 'iloc'):
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold   = y_train.iloc[val_idx]
            else:
                y_train_fold = y_train[train_idx]
                y_val_fold   = y_train[val_idx]

            model = rf.fit(X_train_fold, y_train_fold)
            predictions = rf.predict(X_val_fold)

            accuracy_scores.append(metrics.accuracy_score(y_val_fold, predictions))
            balanced_accuracy_scores.append(metrics.balanced_accuracy_score(y_val_fold, predictions))

            # Choose class to explain (binary: positive class by label order)
            classes = list(model.classes_)
            if len(classes) == 2:
                pos_class = max(classes)  # convention: positive is larger label
                class_index = classes.index(pos_class)
            else:
                class_index = 0  # explain first class; can loop over all if you wish

            # Compute ALE on X_test for each feature using the fixed edges
            for f in feature_list:
                df_ale = ale_1d(model, X_test, f, feature_edges[f], class_index=class_index)
                df_ale['run'] = fold_count
                ale_stack[f].append(df_ale)

    # Aggregate ALE curves: average and std across folds/runs
    ale_curves = {}
    for f in feature_list:
        df_all = pd.concat(ale_stack[f], ignore_index=True)
        grp = df_all.groupby('x', sort=True)
        mean_ale  = grp['ale'].mean()
        std_ale   = grp['ale'].std()
        mean_cnt  = grp['count'].mean()
        ale_curves[f] = pd.DataFrame({
            'x': mean_ale.index.values,
            'ale_mean': mean_ale.values,
            'ale_std': std_ale.values,
            'avg_bin_count': mean_cnt.values
        })

    return ale_curves, accuracy_scores, balanced_accuracy_scores

In [4]:
mu = 'mu_0_4'
algorithm = 'louvain'
feats_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/Node_Features/'
entropies_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/Node_Entropies/'
results_folder = f'LFR_Graph_Data/Community_Data/{algorithm}/results/{mu}'
error = []
us, mode = 0.75, 'node'

# Create results directory if it doesn't exist
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

# Dictionary to store all results
all_results = {}

for feats_fil in os.listdir(feats_folder):
    if feats_fil.endswith(f'{mu}_features.csv'):
        # Extrai o nome do grafo
        graph_name = feats_fil.replace('_features.csv', '')
        
        # Constrói caminhos
        entropies_csv = feats_fil.replace('_features.csv', '_entropies.csv')
        feats_path = os.path.join(feats_folder, feats_fil)
        entropies_path = os.path.join(entropies_folder, entropies_csv)
        individual_results_path = os.path.join(results_folder, f"{graph_name}_results.pkl")
        
        if '_042_' in individual_results_path:
            print('')
        # Pula se já foi processado
        if os.path.exists(individual_results_path):
            print(f"Skipping {graph_name} (already processed)")

            with open(individual_results_path, 'rb') as f:
                result = pickle.load(f)
            all_results[graph_name] = result
            continue
        
        print(f"Processing {graph_name}...")
        
        try:
            # Cria e salva os dados com nome único
            X_train, y_train, X_test, y_test, results_dict = create_and_save_data(
                feats_path, entropies_path, results_folder, us, mode, graph_name
            )
            
            # Treina o modelo
            ale_curves, accuracy_scores, balanced_accuracy_scores = train(
                X_train, X_test, y_train, y_test
            )
            
            # Salva resultados
            graph_results = {
                'ale_curves': ale_curves,
                'accuracy_scores': accuracy_scores,
                'balanced_accuracy_scores': balanced_accuracy_scores,
                'train_test_info': {
                    'X_train_shape': X_train.shape,
                    'X_test_shape': X_test.shape,
                    'y_train_distribution': y_train['Stability'].value_counts().to_dict(),
                    'y_test_distribution': y_test['Stability'].value_counts().to_dict()
                }
            }
            graph_results.update(results_dict)
            
            # Salva resultados individuais
            with open(individual_results_path, 'wb') as fp:
                pickle.dump(graph_results, fp)
            
            all_results[graph_name] = graph_results
            print(f"Completed {graph_name}")
        except Exception as e:
            print(f'Error {e} to process {individual_results_path}...')
            error.append(individual_results_path)
# Save aggregated results 
aggregated_results_path = os.path.join(results_folder, 'all_graphs_results.pkl')
with open(aggregated_results_path, 'wb') as fp:
    pickle.dump(all_results, fp)

print(f"Processed {len(all_results)} graphs")
print(f"Results saved to {aggregated_results_path}")

Skipping graph_0100_mu_0_4 (already processed)
Skipping graph_0101_mu_0_4 (already processed)
Skipping graph_0102_mu_0_4 (already processed)
Skipping graph_0103_mu_0_4 (already processed)
Skipping graph_0104_mu_0_4 (already processed)
Skipping graph_0105_mu_0_4 (already processed)
Skipping graph_0106_mu_0_4 (already processed)
Skipping graph_0107_mu_0_4 (already processed)
Skipping graph_0108_mu_0_4 (already processed)
Skipping graph_0109_mu_0_4 (already processed)
Skipping graph_010_mu_0_4 (already processed)
Skipping graph_0110_mu_0_4 (already processed)
Skipping graph_0111_mu_0_4 (already processed)
Skipping graph_0112_mu_0_4 (already processed)
Skipping graph_0113_mu_0_4 (already processed)
Skipping graph_0114_mu_0_4 (already processed)
Skipping graph_0115_mu_0_4 (already processed)
Skipping graph_0116_mu_0_4 (already processed)
Skipping graph_0117_mu_0_4 (already processed)
Skipping graph_0118_mu_0_4 (already processed)
Skipping graph_0119_mu_0_4 (already processed)
Skipping graph

In [9]:
import os
import pandas as pd

def merge_all_csvs(results_folder):
    """Une todos os CSVs individuais."""
    
    print("=" * 70)
    print(f"📊 UNINDO CSVS - {results_folder}")
    print("=" * 70)
    
    # Lista todos os arquivos
    files = os.listdir(results_folder)
    x_train_files = [f for f in files if f.endswith("_X_train.csv")]
    
    if not x_train_files:
        print("❌ Nenhum arquivo *_X_train.csv encontrado!")
        return None, None, None, None
    
    print(f"📁 Encontrados {len(x_train_files)} arquivos")
    
    X_train_all = pd.DataFrame()
    y_train_all = pd.DataFrame()
    X_test_all = pd.DataFrame()
    y_test_all = pd.DataFrame()
    
    for file in x_train_files:
        graph_name = file.replace('_X_train.csv', '')
        
        X_train = pd.read_csv(os.path.join(results_folder, file), index_col=0)
        y_train = pd.read_csv(os.path.join(results_folder, f"{graph_name}_y_train.csv"), index_col=0)
        
        X_train_all = pd.concat([X_train_all, X_train])
        y_train_all = pd.concat([y_train_all, y_train])
        
        # Tenta carregar X_test
        test_file = os.path.join(results_folder, f"{graph_name}_X_test.csv")
        if os.path.exists(test_file):
            X_test = pd.read_csv(test_file, index_col=0)
            y_test = pd.read_csv(os.path.join(results_folder, f"{graph_name}_y_test.csv"), index_col=0)
            X_test_all = pd.concat([X_test_all, X_test])
            y_test_all = pd.concat([y_test_all, y_test])
    
    # Salva
    X_train_all.to_csv(os.path.join(results_folder, "all_X_train.csv"))
    y_train_all.to_csv(os.path.join(results_folder, "all_y_train.csv"))
    X_test_all.to_csv(os.path.join(results_folder, "all_X_test.csv"))
    y_test_all.to_csv(os.path.join(results_folder, "all_y_test.csv"))
    
    print(f"\n✅ all_X_train.csv: {X_train_all.shape[0]:,} amostras")
    print(f"✅ all_y_train.csv: {y_train_all.shape[0]:,} amostras")
    if not X_test_all.empty:
        print(f"✅ all_X_test.csv: {X_test_all.shape[0]:,} amostras")
        print(f"✅ all_y_test.csv: {y_test_all.shape[0]:,} amostras")
    
    return X_train_all, y_train_all, X_test_all, y_test_all

In [7]:
results_folder = "LFR_Graph_Data/Community_Data/lpa/results/mu_0_3"

merge_all_csvs(results_folder)

📊 UNINDO CSVS - LFR_Graph_Data/Community_Data/lpa/results/mu_0_3
📁 Encontrados 120 arquivos

✅ all_X_train.csv: 105,216 amostras
✅ all_y_train.csv: 105,216 amostras
✅ all_X_test.csv: 26,361 amostras
✅ all_y_test.csv: 26,361 amostras


(                            Degree  Clustering Coefficient  Betweenness  \
 graph_0100_mu_0_3_node_120      54                0.102725     0.003266   
 graph_0100_mu_0_3_node_123      20                0.105263     0.000689   
 graph_0100_mu_0_3_node_969      37                0.124625     0.001720   
 graph_0100_mu_0_3_node_759      17                0.139706     0.000381   
 graph_0100_mu_0_3_node_69       27                0.153846     0.000747   
 ...                            ...                     ...          ...   
 graph_042_mu_0_3_node_919       21                0.100000     0.000576   
 graph_042_mu_0_3_node_294       19                0.087719     0.000494   
 graph_042_mu_0_3_node_509       16                0.241667     0.000334   
 graph_042_mu_0_3_node_128       26                0.129231     0.001150   
 graph_042_mu_0_3_node_888       16                0.175000     0.000325   
 
                             Closeness  Shortest Path  Eigenvector    E In  \
 graph_0